# Phase 7: Network Construction & Advanced Analysis

<div class='alert alert-info'>
In this final notebook, we build the actual collaboration network using NetworkX. We will analyze  properties like Network Density and Connected Components, and then dive into node metrics such as Degree Centrality. Finally, we apply the Louvain method to detect distinct communities (sub-graphs) within the Iranian cinema ecosystem.
</div>

In [1]:
import networkx as nx
import pandas as pd

In [2]:
df_pairs = pd.read_parquet('../data/processed/collaboration_pairs.parquet')
print(df_pairs.shape)

(301594, 6)


In [3]:
G = nx.Graph()

for _, row in df_pairs.iterrows():
    if G.has_edge(row['person1'], row['person2']):
        G[row['person1']][row['person2']]['weight'] += row['weight']
    else:
        G.add_edge(row['person1'], row['person2'], weight=row['weight'])

print("number of nodes(individuals)", G.number_of_nodes())
print("number of edges(collaborations)", G.number_of_edges())

number of nodes(individuals) 15716
number of edges(collaborations) 258915


<div class='alert alert-warning'>
Step 1: Macroscopic Network Properties
We calculate the overall <strong>Density</strong> of the network to see how tightly knit the industry is. We also identify the <strong>Connected Components</strong>. Since real-world networks often have fragmented parts, isolating the <strong>Giant Component</strong> (the largest connected sub-graph) is crucial for advanced centrality and community detection algorithms.
</div>

In [4]:
# Network Denisty
density = nx.density(G)
print("Density:", density)

# Connected Components
components = list(nx.connected_components(G))
components_sorted = sorted(components, key=len, reverse=True)
print("number of connected components: ", len(components))
print("Giant Component of the network: ", len(components_sorted[0]))
print("percentage of the total network: ", len(components_sorted[0]) / G.number_of_nodes())

Density: 0.0020966734789085975
number of connected components:  241
Giant Component of the network:  14305
percentage of the total network:  0.9102188852125223


<div class='alert alert-warning'>
Step 2: Degree Centrality
Who are the most central hubs in the industry? By calculating the Degree (and Weighted Degree) centrality, we can identify the individuals who act as the primary structural pillars of the Iranian cinema network.
</div>

In [5]:
# Degree و Weighted Degree
degree_dict = dict(G.degree())
weighted_degree_dict = dict(G.degree(weight='weight'))

top_degree = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)[:10]
print("the most central hubs: ", top_degree)

the most central hubs:  [('nm2465620', 829), ('nm1714957', 772), ('nm1286513', 766), ('nm5747379', 761), ('nm0451571', 742), ('nm0707308', 736), ('nm2932763', 720), ('nm0008206', 712), ('nm2114785', 708), ('nm1471776', 677)]


In [ ]:
# Betweenness Centrality (روی Giant Component، چون محاسبه‌ش سنگینه)
giant = G.subgraph(components_sorted[0])
betweenness = nx.betweenness_centrality(giant, weight='weight')
top_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
print("Bridge :", top_betweenness)

<div class='alert alert-warning'>
Step 3: Community Detection (Louvain Method)
To uncover hidden factions or highly collaborative sub-groups within the Giant Component, we apply the <strong>Louvain Modularity</strong> algorithm. This helps us see if the industry naturally partitions into distinct "families" or professional cliques.
</div>

In [ ]:
pip install python-louvain --break-system-packages

In [ ]:
import community as community_louvain

partition = community_louvain.best_partition(giant, weight='weight')
print("number of communities: ", len(set(partition.values())))